# LeetCode 653 — Two Sum IV - Input is a BST bằng Python

**Link đề:** https://leetcode.com/problems/two-sum-iv-input-is-a-bst/
**Độ khó:** Easy · **Chủ đề:** Hash Table, Two Pointers, Tree, DFS, BFS, Binary Search Tree

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findTarget(self, root: Optional[TreeNode], k: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu cây nhị phân (`TreeNode`), cây nhị phân tìm kiếm (**BST**) và cách dựng cây từ list kiểu LeetCode.
2. Duyệt cây bằng **DFS đệ quy**, **DFS bằng stack**, **BFS bằng queue**.
3. Áp dụng lại ý tưởng **Two Sum + hash set**: "đã thấy `k - x` chưa?".
4. Khai thác tính chất BST: **duyệt in-order cho dãy tăng** → dùng **hai con trỏ**.
5. So sánh 4 cách: brute force, hash set, in-order + two pointers, tìm kiếm trên BST.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho gốc `root` của một **cây nhị phân tìm kiếm** và số nguyên `k`. Trả `True` nếu tồn tại **hai phần tử khác nhau** trong cây có tổng bằng `k`, ngược lại `False`.

**Ví dụ 1:** `root = [5,3,6,2,4,null,7]`, `k = 9` → `True` (ví dụ `2 + 7`, `3 + 6`, `5 + 4`).
**Ví dụ 2:** cùng cây, `k = 28` → `False`.

```text
        5
       / \
      3   6
     / \    \
    2   4    7
```

**Constraints:**
- Số nút trong `[1, 10^4]`.
- `-10^4 <= Node.val <= 10^4`.
- `root` chắc chắn là BST hợp lệ.
- `-10^5 <= k <= 10^5`.

Chú ý "hai phần tử": một nút **không được dùng hai lần** — cây chỉ có nút `5` với `k = 10` phải trả `False`.

## 2. Nền tảng: `TreeNode`, BST và dựng cây từ list

### 2.1. Lớp `TreeNode`
Mỗi nút có `val`, con trái `left`, con phải `right` (có thể là `None`).
LeetCode biểu diễn cây bằng list **theo tầng** (level order), `None` (`null`) là chỗ trống.

Ta tự viết hàm `build_tree(values)` để notebook chạy độc lập, và `Optional[TreeNode]` nghĩa là "TreeNode hoặc None".

In [1]:
from typing import Optional
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values: list) -> Optional[TreeNode]:
    # Dựng cây từ list theo tầng kiểu LeetCode, None = không có nút.
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def print_tree(node: Optional[TreeNode], prefix: str = "", label: str = "root") -> None:
    # In cây xoay ngang: mỗi tầng thụt vào thêm 4 khoảng trắng.
    if node is None:
        return
    print(prefix + f"{label}: {node.val}")
    print_tree(node.left, prefix + "    ", "L")
    print_tree(node.right, prefix + "    ", "R")


root = build_tree([5, 3, 6, 2, 4, None, 7])
print_tree(root)

root: 5
    L: 3
        L: 2
        R: 4
    R: 6
        R: 7


### 2.2. Cây nhị phân tìm kiếm (BST) là gì?

Với **mọi** nút `x`:
- mọi giá trị ở cây con trái **nhỏ hơn** `x.val`,
- mọi giá trị ở cây con phải **lớn hơn** `x.val`.

Hệ quả quan trọng: **duyệt in-order** (trái → gốc → phải) cho ra dãy **tăng dần**.

In [2]:
def inorder(node: Optional[TreeNode], out: list) -> list:
    if node is None:            # điều kiện dừng của đệ quy
        return out
    inorder(node.left, out)     # 1) trái
    out.append(node.val)        # 2) gốc
    inorder(node.right, out)    # 3) phải
    return out

print("in-order:", inorder(root, []))

in-order: [2, 3, 4, 5, 6, 7]


### 2.3. Đệ quy (recursion) trong 30 giây
Một hàm **gọi lại chính nó** trên bài toán nhỏ hơn (cây con). Luôn cần **điều kiện dừng** (`node is None`).
Mỗi lời gọi được đặt lên **call stack**; độ sâu tối đa = chiều cao cây `h`.

In [3]:
def count_nodes(node: Optional[TreeNode]) -> int:
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

def height(node: Optional[TreeNode]) -> int:
    if node is None:
        return 0
    return 1 + max(height(node.left), height(node.right))

print("số nút =", count_nodes(root), "| chiều cao =", height(root))

số nút = 6 | chiều cao = 3


### 2.4. `set` và lại ý tưởng Two Sum
Two Sum cổ điển: duyệt từng `x`, hỏi "`k - x` đã gặp chưa?" (tra `set` trung bình `O(1)`), nếu chưa thì thêm `x` vào `set`.
**Kiểm tra trước, thêm sau** → không bao giờ dùng một phần tử hai lần.

In [4]:
def two_sum_exists(values: list[int], k: int) -> bool:
    seen = set()
    for x in values:
        if k - x in seen:
            return True
        seen.add(x)
    return False

print(two_sum_exists([5, 3, 6, 2, 4, 7], 9))    # True
print(two_sum_exists([5], 10))                  # False: không dùng 5 hai lần

True
False


### 2.5. BFS với `deque`
`deque` (hàng đợi hai đầu) có `append` và `popleft` đều `O(1)`. BFS lấy nút ở đầu hàng đợi, đẩy con vào cuối → duyệt **theo tầng**.

In [5]:
def bfs_values(root: Optional[TreeNode]) -> list[int]:
    out = []
    q = deque([root]) if root else deque()
    while q:
        node = q.popleft()
        out.append(node.val)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    return out

print("BFS theo tầng:", bfs_values(root))

BFS theo tầng: [5, 3, 6, 2, 4, 7]


## 3. Cách trực giác — Brute Force

1. Gom tất cả giá trị trong cây vào một list.
2. Thử mọi cặp `i < j`, nếu `vals[i] + vals[j] == k` → `True`.

Time `O(n²)` — với `n = 10^4` khoảng `5·10^7` cặp, chậm trong Python.

In [6]:
def find_target_bruteforce(root: Optional[TreeNode], k: int) -> bool:
    vals = inorder(root, [])
    n = len(vals)
    for i in range(n):
        for j in range(i + 1, n):
            if vals[i] + vals[j] == k:
                return True
    return False

print(find_target_bruteforce(root, 9), find_target_bruteforce(root, 28))

True False


## 4. Nút thắt và insight

**Nút thắt:** với mỗi `x`, ta quét lại toàn bộ để tìm `k - x`.

**Insight A (hash set — không cần BST):** duyệt cây một lần, với mỗi nút hỏi `k - val in seen`. Đây đúng là Two Sum trên cây. `O(n)` time, `O(n)` space.

**Insight B (dùng tính chất BST):** in-order cho list **đã sắp xếp** → áp dụng **hai con trỏ** `left`, `right`:
- tổng `< k` → cần lớn hơn → `left += 1`;
- tổng `> k` → cần nhỏ hơn → `right -= 1`;
- tổng `== k` → `True`.

**Insight C (tìm kiếm trên BST):** với mỗi nút `x`, đi từ gốc tìm `k - x` theo quy tắc BST trong `O(h)`. Tổng `O(n·h)`, không cần bộ nhớ phụ ngoài đệ quy.

In [7]:
k = 9
for x in [2, 3, 4, 5, 6, 7]:
    print(f"x={x} cần tìm k - x = {k - x}")

x=2 cần tìm k - x = 7
x=3 cần tìm k - x = 6
x=4 cần tìm k - x = 5
x=5 cần tìm k - x = 4
x=6 cần tìm k - x = 3
x=7 cần tìm k - x = 2


## 5. Lời giải chính — DFS + hash set (giảng từng đoạn)

**Đoạn 1:** tạo `seen = set()`.

**Đoạn 2:** viết hàm đệ quy `dfs(node)`:
- `node is None` → `False` (không có gì để tìm).
- nếu `k - node.val in seen` → `True` (tìm thấy cặp).
- thêm `node.val` vào `seen`.
- trả `dfs(node.left) or dfs(node.right)`.

**Đoạn 3:** gọi `dfs(root)`.

Toán tử `or` trong Python **đoản mạch** (short-circuit): nếu vế trái đã `True` thì vế phải không chạy → dừng sớm.

In [8]:
print(True or print("vế phải KHÔNG chạy"))
print(False or "vế phải chạy")

True
vế phải chạy


In [9]:
def find_target_hashset(root: Optional[TreeNode], k: int) -> bool:
    seen = set()                                   # đoạn 1

    def dfs(node: Optional[TreeNode]) -> bool:     # đoạn 2
        if node is None:
            return False
        if k - node.val in seen:
            return True
        seen.add(node.val)
        return dfs(node.left) or dfs(node.right)

    return dfs(root)                               # đoạn 3

print(find_target_hashset(root, 9))   # True
print(find_target_hashset(root, 28))  # False
print(find_target_hashset(build_tree([5]), 10))  # False

True
False
False


## 6. TRACE từng bước

Cây `[5,3,6,2,4,null,7]`, `k = 9`. DFS đi theo thứ tự **tiền thứ tự** (pre-order): gốc → trái → phải.

| Bước | Nút | `k - val` | `seen` trước | Quyết định |
|---:|---:|---:|---|---|
| 1 | 5 | 4 | `{}` | chưa thấy 4 → thêm 5 |
| 2 | 3 | 6 | `{5}` | chưa thấy 6 → thêm 3 |
| 3 | 2 | 7 | `{5,3}` | chưa thấy 7 → thêm 2 |
| 4 | 4 | 5 | `{5,3,2}` | **thấy 5** → `True` |

Dừng ngay ở nút 4 nhờ `or` đoản mạch; nhánh phải (6, 7) không bao giờ được thăm.

In [10]:
def trace_find_target(root: Optional[TreeNode], k: int) -> bool:
    seen = set()
    step = [0]

    def dfs(node, depth):
        if node is None:
            return False
        step[0] += 1
        need = k - node.val
        indent = "  " * depth
        print(f"{indent}bước {step[0]}: nút {node.val}, cần {need}, seen trước = {sorted(seen)}", end="")
        if need in seen:
            print(" -> TÌM THẤY")
            return True
        seen.add(node.val)
        print(f" -> thêm {node.val}")
        return dfs(node.left, depth + 1) or dfs(node.right, depth + 1)

    ans = dfs(root, 0)
    print("Kết quả:", ans)
    return ans

trace_find_target(root, 9)
print()
trace_find_target(root, 28)

bước 1: nút 5, cần 4, seen trước = [] -> thêm 5
  bước 2: nút 3, cần 6, seen trước = [5] -> thêm 3
    bước 3: nút 2, cần 7, seen trước = [3, 5] -> thêm 2
    bước 4: nút 4, cần 5, seen trước = [2, 3, 5] -> TÌM THẤY
Kết quả: True

bước 1: nút 5, cần 23, seen trước = [] -> thêm 5
  bước 2: nút 3, cần 25, seen trước = [5] -> thêm 3
    bước 3: nút 2, cần 26, seen trước = [3, 5] -> thêm 2
    bước 4: nút 4, cần 24, seen trước = [2, 3, 5] -> thêm 4
  bước 5: nút 6, cần 22, seen trước = [2, 3, 4, 5] -> thêm 6
    bước 6: nút 7, cần 21, seen trước = [2, 3, 4, 5, 6] -> thêm 7
Kết quả: False


False

## 7. Code nộp LeetCode

### 7.1. Cách 1 — DFS + hash set (khuyên dùng)
Trên LeetCode, `TreeNode` và `Optional` đã có sẵn; ở notebook ta đã định nghĩa ở mục 2.

In [11]:
class Solution:
    def findTarget(self, root: Optional[TreeNode], k: int) -> bool:
        seen = set()

        def dfs(node: Optional[TreeNode]) -> bool:
            if node is None:
                return False
            if k - node.val in seen:
                return True
            seen.add(node.val)
            return dfs(node.left) or dfs(node.right)

        return dfs(root)

print(Solution().findTarget(build_tree([5, 3, 6, 2, 4, None, 7]), 9))
print(Solution().findTarget(build_tree([5, 3, 6, 2, 4, None, 7]), 28))

True
False


### 7.2. Cách 1b — BFS + hash set (không đệ quy)
Cây lệch có thể sâu tới `10^4` tầng — vượt giới hạn đệ quy mặc định của Python (~1000). Dùng hàng đợi giúp an toàn.

In [12]:
class SolutionBFS:
    def findTarget(self, root: Optional[TreeNode], k: int) -> bool:
        seen = set()
        q = deque([root])
        while q:
            node = q.popleft()
            if node is None:
                continue
            if k - node.val in seen:
                return True
            seen.add(node.val)
            q.append(node.left)
            q.append(node.right)
        return False

print(SolutionBFS().findTarget(build_tree([5, 3, 6, 2, 4, None, 7]), 9))

True


### 7.3. Cách 2 — In-order + hai con trỏ (dùng tính chất BST)

**Trace hai con trỏ** trên `vals = [2, 3, 4, 5, 6, 7]`, `k = 9`:

| `l` | `r` | `vals[l] + vals[r]` | Hành động |
|---:|---:|---:|---|
| 0 | 5 | 2 + 7 = 9 | bằng `k` → `True` |

Với `k = 28`: tổng lớn nhất `6 + 7 = 13 < 28` → `l` tăng dần đến khi gặp `r` → `False`.

In [13]:
def trace_two_pointers(vals: list[int], k: int) -> bool:
    l, r = 0, len(vals) - 1
    while l < r:
        s = vals[l] + vals[r]
        action = "TÌM THẤY" if s == k else ("l += 1" if s < k else "r -= 1")
        print(f"l={l}({vals[l]}), r={r}({vals[r]}), tổng={s} -> {action}")
        if s == k:
            return True
        if s < k:
            l += 1
        else:
            r -= 1
    print("l gặp r -> False")
    return False

vals = inorder(root, [])
trace_two_pointers(vals, 9)
print()
trace_two_pointers(vals, 28)

l=0(2), r=5(7), tổng=9 -> TÌM THẤY

l=0(2), r=5(7), tổng=9 -> l += 1
l=1(3), r=5(7), tổng=10 -> l += 1
l=2(4), r=5(7), tổng=11 -> l += 1
l=3(5), r=5(7), tổng=12 -> l += 1
l=4(6), r=5(7), tổng=13 -> l += 1
l gặp r -> False


False

In [14]:
class SolutionTwoPointers:
    def findTarget(self, root: Optional[TreeNode], k: int) -> bool:
        vals = []
        def inorder(node):
            if node:
                inorder(node.left)
                vals.append(node.val)
                inorder(node.right)
        inorder(root)
        l, r = 0, len(vals) - 1
        while l < r:
            s = vals[l] + vals[r]
            if s == k:
                return True
            if s < k:
                l += 1
            else:
                r -= 1
        return False

print(SolutionTwoPointers().findTarget(build_tree([5, 3, 6, 2, 4, None, 7]), 9))
print(SolutionTwoPointers().findTarget(build_tree([5, 3, 6, 2, 4, None, 7]), 28))

True
False


### 7.4. Cách 3 — Với mỗi nút, tìm `k - x` trên BST

Tìm một giá trị `t` trên BST: bắt đầu ở gốc, nếu `t < node.val` đi trái, `t > node.val` đi phải, bằng thì thấy. Mỗi lần tìm `O(h)`.

Bẫy: nếu `k - x == x` thì sẽ tìm thấy **chính nút đó** → phải kiểm tra nút tìm được **khác** nút hiện tại (`found is not node`).

In [15]:
def bst_search(root: Optional[TreeNode], t: int) -> Optional[TreeNode]:
    node = root
    while node:
        if t == node.val:
            return node
        node = node.left if t < node.val else node.right
    return None

print("tìm 4:", bst_search(root, 4), "| tìm 8:", bst_search(root, 8))


class SolutionSearch:
    def findTarget(self, root: Optional[TreeNode], k: int) -> bool:
        def dfs(node):
            if node is None:
                return False
            found = bst_search(root, k - node.val)
            if found is not None and found is not node:
                return True
            return dfs(node.left) or dfs(node.right)
        return dfs(root)

print(SolutionSearch().findTarget(root, 9), SolutionSearch().findTarget(root, 10), SolutionSearch().findTarget(build_tree([5]), 10))

tìm 4: TreeNode(4) | tìm 8: None
True True False


## 8. Vì sao đúng?

**Hash set — bất biến:** tại thời điểm thăm nút `y`, `seen` chứa đúng giá trị của các nút đã thăm **trước** `y`.
- Nếu tồn tại cặp nút `(a, b)` với `a.val + b.val = k`, gọi `b` là nút được thăm **sau**. Khi thăm `b`, `a.val` đã có trong `seen` → trả `True`. (Có thể dừng sớm hơn ở một cặp khác — vẫn đúng.)
- Nếu trả `True` ở nút `y`, thì `k - y.val` thuộc một nút **khác** đã thăm trước → cặp hợp lệ. Vì kiểm tra trước khi thêm, không bao giờ ghép `y` với chính nó.

**Hai con trỏ — bất biến:** mọi cặp chứa chỉ số `< l` hoặc `> r` đều đã bị loại. Khi `vals[l] + vals[r] < k`, `vals[l]` cộng với **bất kỳ** phần tử nào `≤ vals[r]` cũng `< k` → loại `l` an toàn. Tương tự khi `> k` loại `r`.

## 9. Độ phức tạp và so sánh

Gọi `n` = số nút, `h` = chiều cao (`log n` nếu cân bằng, tới `n` nếu lệch).

| Cách | Time | Space thêm | Dùng tính chất BST? |
|---|---:|---:|:---:|
| Brute force mọi cặp | `O(n²)` | `O(n)` | Không |
| DFS/BFS + hash set | `O(n)` trung bình | `O(n)` | Không |
| In-order + 2 con trỏ | `O(n)` | `O(n)` | **Có** |
| Tìm `k-x` trên BST | `O(n·h)` | `O(h)` | **Có** |

Ghi chú: cách hash set áp dụng cho **mọi cây nhị phân**, không chỉ BST.
Có thêm cách nâng cao: hai **iterator BST** (một tiến, một lùi) dùng stack → `O(n)` time, `O(h)` space.

## 10. Bộ test

1. Ví dụ đề.
2. Một nút (không được tự cộng với chính nó).
3. Giá trị âm, `k` âm.
4. Cây lệch (như danh sách liên kết).
5. Ngẫu nhiên: sinh BST từ tập số ngẫu nhiên, so mọi cách với brute force.

In [16]:
import random

def insert_bst(root, v):
    if root is None:
        return TreeNode(v)
    if v < root.val:
        root.left = insert_bst(root.left, v)
    else:
        root.right = insert_bst(root.right, v)
    return root

def bst_from_values(values):
    r = None
    for v in values:
        r = insert_bst(r, v)
    return r

solvers = [Solution().findTarget, SolutionBFS().findTarget, SolutionTwoPointers().findTarget,
           SolutionSearch().findTarget, find_target_bruteforce]
cases = [
    (build_tree([5, 3, 6, 2, 4, None, 7]), 9, True),
    (build_tree([5, 3, 6, 2, 4, None, 7]), 28, False),
    (build_tree([5]), 10, False),
    (build_tree([2, 1, 3]), 4, True),
    (build_tree([2, 1, 3]), 1, False),
    (build_tree([0, -1, 2, -3, None, None, 4]), -4, True),     # -1 + -3
    (bst_from_values([1, 2, 3, 4, 5]), 9, True),               # cây lệch phải
    (bst_from_values([5, 4, 3, 2, 1]), 2, False),
]
for i, (tree, k, expected) in enumerate(cases):
    for f in solvers:
        assert f(tree, k) == expected, (i, f, k)
    print(f"case {i}: k={k:>3} -> {expected} PASS")

case 0: k=  9 -> True PASS
case 1: k= 28 -> False PASS
case 2: k= 10 -> False PASS
case 3: k=  4 -> True PASS
case 4: k=  1 -> False PASS
case 5: k= -4 -> True PASS
case 6: k=  9 -> True PASS
case 7: k=  2 -> False PASS


In [17]:
rng = random.Random(653)
for _ in range(300):
    vals = rng.sample(range(-30, 31), rng.randint(1, 12))
    tree = bst_from_values(vals)
    k = rng.randint(-60, 60)
    expected = find_target_bruteforce(tree, k)
    for f in solvers:
        assert f(tree, k) == expected
print("300 cây BST ngẫu nhiên: mọi cách khớp brute force PASS")

300 cây BST ngẫu nhiên: mọi cách khớp brute force PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Thêm vào `seen` **trước** khi kiểm tra
Cây chỉ có `5`, `k = 10`: thêm 5 rồi hỏi `10 - 5 = 5 in seen` → `True` sai.

In [18]:
def wrong_add_first(root, k):
    seen = set()
    def dfs(node):
        if node is None:
            return False
        seen.add(node.val)                 # SAI: thêm trước
        if k - node.val in seen:
            return True
        return dfs(node.left) or dfs(node.right)
    return dfs(root)

print("sai:", wrong_add_first(build_tree([5]), 10), "| đúng:", Solution().findTarget(build_tree([5]), 10))

sai: True | đúng: False


### Lỗi 2: Chỉ so sánh cha–con
Một số bạn chỉ kiểm tra `node.val + node.left.val == k` → bỏ sót cặp ở hai nhánh xa nhau (như `2 + 7`).

### Lỗi 3: Quên `return` kết quả đệ quy
Viết `dfs(node.left); dfs(node.right)` mà không `return dfs(...) or dfs(...)` → kết quả `True` ở sâu bên dưới bị mất, hàm trả `None`.

In [19]:
def wrong_no_return(root, k):
    seen = set()
    def dfs(node):
        if node is None:
            return False
        if k - node.val in seen:
            return True
        seen.add(node.val)
        dfs(node.left)                     # SAI: bỏ mất giá trị trả về
        dfs(node.right)
    return dfs(root)

print("sai:", wrong_no_return(root, 9), "| đúng:", Solution().findTarget(root, 9))

sai: None | đúng: True


### Lỗi 4: Hai con trỏ với `while l <= r`
Khi `l == r` ta đang cộng một phần tử với chính nó → có thể trả `True` sai (ví dụ `[2, 5]`, `k = 10`).

### Lỗi 5: Đệ quy quá sâu
Cây lệch 10^4 nút → `RecursionError` trong Python nếu không tăng giới hạn. Dùng BFS/stack để an toàn.

In [20]:
def wrong_le(vals, k):
    l, r = 0, len(vals) - 1
    while l <= r:                          # SAI
        s = vals[l] + vals[r]
        if s == k: return True
        if s < k: l += 1
        else: r -= 1
    return False

print("sai:", wrong_le([2, 5], 10), "| đúng:", trace_two_pointers([2, 5], 10))

l=0(2), r=1(5), tổng=7 -> l += 1
l gặp r -> False
sai: True | đúng: False


## 12. Mẹo nhận dạng pattern

- "Có **hai phần tử** tổng bằng `k`?" → **Two Sum**: hash set (không cần thứ tự) hoặc hai con trỏ (khi dữ liệu **đã sắp xếp**).
- "Input là **BST**" → nhớ ngay: **in-order = dãy tăng**; tìm kiếm `O(h)`.
- Bài cây chỉ cần "thăm hết các nút" → chọn DFS hoặc BFS tùy thích; cây sâu thì tránh đệ quy.
- Các bài cùng họ: 1 Two Sum, 167 Two Sum II (mảng đã sắp xếp), 170 Two Sum III (thiết kế), 1214 Two Sum BSTs.

## 13. Biến thể

### 13.1. Trả về chính cặp giá trị

In [21]:
def find_pair(root: Optional[TreeNode], k: int):
    seen = set()
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if k - node.val in seen:
            return (k - node.val, node.val)
        seen.add(node.val)
        stack.append(node.right)
        stack.append(node.left)
    return None

print(find_pair(root, 9), find_pair(root, 13), find_pair(root, 28))

(5, 4) (6, 7) None


### 13.2. Two Sum BSTs (LeetCode 1214, Premium): một số từ cây A, một số từ cây B
Gom giá trị cây A vào set, duyệt cây B kiểm tra `k - val in setA`.

In [22]:
def two_sum_bsts(root1, root2, target):
    set_a = set(inorder(root1, []))
    return any(target - v in set_a for v in inorder(root2, []))

print(two_sum_bsts(build_tree([2, 1, 4]), build_tree([1, 0, 3]), 5))   # True (2+3 hoặc 4+1)
print(two_sum_bsts(build_tree([0, -10, 10]), build_tree([5, 1, 7, 0, 2]), 18))  # False

True
False


## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Đoán output:** in-order của cây `[8, 3, 10, 1, 6, None, 14]` là gì?

**Bài 2 — Trace:** chạy tay cách hai con trỏ trên in-order của cây Bài 1 với `k = 16`.

**Bài 3 — TODO:** viết `findTarget` bằng **DFS dùng stack** (không đệ quy) + set.

In [23]:
def exercise_dfs_stack(root: Optional[TreeNode], k: int) -> bool:
    # TODO 1: stack = [root], seen = set()
    # TODO 2: pop nút; bỏ qua None; kiểm tra k - val in seen; thêm val; đẩy 2 con
    pass

# assert exercise_dfs_stack(build_tree([5, 3, 6, 2, 4, None, 7]), 9) is True
# assert exercise_dfs_stack(build_tree([5]), 10) is False

**Bài 4 — TODO:** đếm **số cặp** (không thứ tự) có tổng bằng `k` trong BST (giá trị phân biệt).

In [24]:
def exercise_count_pairs(root: Optional[TreeNode], k: int) -> int:
    # TODO: in-order -> hai con trỏ; khi tổng == k thì count += 1 và dịch CẢ hai con trỏ
    pass

# assert exercise_count_pairs(build_tree([5, 3, 6, 2, 4, None, 7]), 9) == 3

**Bài 5 — TODO:** kiểm tra cây nhị phân bất kỳ có phải BST không (gợi ý: in-order phải tăng nghiêm ngặt).

In [25]:
def exercise_is_bst(root: Optional[TreeNode]) -> bool:
    # TODO: lấy in-order, kiểm tra vals[i] < vals[i+1] với mọi i
    pass

# assert exercise_is_bst(build_tree([2, 1, 3])) is True
# assert exercise_is_bst(build_tree([5, 1, 4, None, None, 3, 6])) is False

**Bài 6 — Mini challenge:** cài `findTarget` với **hai iterator BST** (một đi tăng, một đi giảm) dùng stack, bộ nhớ `O(h)`.

In [26]:
def exercise_two_iterators(root: Optional[TreeNode], k: int) -> bool:
    # TODO: viết hàm push_left / push_right; lấy next nhỏ nhất và next lớn nhất; dịch như hai con trỏ
    pass

# assert exercise_two_iterators(build_tree([5, 3, 6, 2, 4, None, 7]), 9) is True
# assert exercise_two_iterators(build_tree([5, 3, 6, 2, 4, None, 7]), 28) is False

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | In-order BST luôn tăng | Liệt kê trái → gốc → phải | 6 phần tử, bắt đầu bằng 1 |
| 2 | `l = 0`, `r = 5` | So tổng với 16 | `1+14 < 16` → l tăng, `3+14 > 16` → r giảm... |
| 3 | Giống BFS nhưng dùng `list.pop()` | `if node is None: continue` | đẩy `node.right` rồi `node.left` |
| 4 | Không `return` khi tìm thấy | Dịch cả `l` và `r` sau khi đếm | Giá trị phân biệt → không có cặp trùng |
| 5 | Dùng `inorder` có sẵn | `all(...)` với `zip(vals, vals[1:])` | `a < b` cho mọi cặp kề |
| 6 | Stack trái chứa đường đi về min | Lấy nút xong thì push trái của `node.right` | Đối xứng cho iterator giảm |

## 16. Đáp án tham khảo

**Bài 1:** `[1, 3, 6, 8, 10, 14]`.

**Bài 2:** `1+14=15<16` → l=1; `3+14=17>16` → r=4; `3+10=13<16` → l=2; `6+10=16` → `True`.

In [27]:
t1 = build_tree([8, 3, 10, 1, 6, None, 14])
print("Bài 1:", inorder(t1, []))
print("Bài 2:")
trace_two_pointers(inorder(t1, []), 16)

Bài 1: [1, 3, 6, 8, 10, 14]
Bài 2:
l=0(1), r=5(14), tổng=15 -> l += 1
l=1(3), r=5(14), tổng=17 -> r -= 1
l=1(3), r=4(10), tổng=13 -> l += 1
l=2(6), r=4(10), tổng=16 -> TÌM THẤY


True

In [28]:
def answer_dfs_stack(root, k):
    seen, stack = set(), [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if k - node.val in seen:
            return True
        seen.add(node.val)
        stack.append(node.right)
        stack.append(node.left)
    return False


def answer_count_pairs(root, k):
    vals = inorder(root, [])
    l, r, count = 0, len(vals) - 1, 0
    while l < r:
        s = vals[l] + vals[r]
        if s == k:
            count += 1
            l += 1
            r -= 1
        elif s < k:
            l += 1
        else:
            r -= 1
    return count


def answer_is_bst(root):
    vals = inorder(root, [])
    return all(a < b for a, b in zip(vals, vals[1:]))


def answer_two_iterators(root, k):
    lo_stack, hi_stack = [], []

    def push_left(node):
        while node:
            lo_stack.append(node)
            node = node.left

    def push_right(node):
        while node:
            hi_stack.append(node)
            node = node.right

    push_left(root)
    push_right(root)
    lo, hi = lo_stack.pop(), hi_stack.pop()
    push_left(lo.right)
    push_right(hi.left)
    while lo is not hi and lo.val < hi.val:
        s = lo.val + hi.val
        if s == k:
            return True
        if s < k:
            lo = lo_stack.pop()
            push_left(lo.right)
        else:
            hi = hi_stack.pop()
            push_right(hi.left)
    return False


tree = build_tree([5, 3, 6, 2, 4, None, 7])
assert answer_dfs_stack(tree, 9) is True
assert answer_dfs_stack(build_tree([5]), 10) is False
assert answer_count_pairs(tree, 9) == 3
assert answer_is_bst(build_tree([2, 1, 3])) is True
assert answer_is_bst(build_tree([5, 1, 4, None, None, 3, 6])) is False
assert answer_two_iterators(tree, 9) is True
assert answer_two_iterators(tree, 28) is False
rng = random.Random(6530)
for _ in range(300):
    t = bst_from_values(rng.sample(range(-20, 21), rng.randint(1, 10)))
    kk = rng.randint(-40, 40)
    exp = find_target_bruteforce(t, kk)
    assert answer_dfs_stack(t, kk) == exp and answer_two_iterators(t, kk) == exp
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findTarget(self, root: Optional[TreeNode], k: int) -> bool:
        seen = set()
        def dfs(node):
            if node is None:
                return False
            if k - node.val in seen:     # kiểm tra TRƯỚC
                return True
            seen.add(node.val)           # thêm SAU
            return dfs(node.left) or dfs(node.right)
        return dfs(root)
```

**Câu thần chú:** *"Two Sum trên cây = duyệt cây + set; BST thì in-order + hai con trỏ."*

**Checklist:**
- [ ] Kiểm tra `k - val in seen` **trước** `seen.add(val)`.
- [ ] `return dfs(left) or dfs(right)` — không bỏ giá trị trả về.
- [ ] Hai con trỏ dùng `while l < r` (không `<=`).
- [ ] Nghĩ tới cây một nút và cây lệch (đệ quy sâu).
- [ ] Time `O(n)`, Space `O(n)`.